In [45]:
from openai import OpenAI
import pdb
from typing import List, Dict, Tuple, Optional, Union, Callable, Generator, Iterable
from abc import ABC, abstractmethod
import os

class GenerateResponse(ABC):
    @abstractmethod
    def __call__(self, prefix:str, queries: List[str], **kwargs)->List[Dict[str, str]]:
        pass
    
class OpenAiGenerateResponse(GenerateResponse):
    client: OpenAI
    model: str
    system_prompt: str
    total_prompt_tokens: int
    total_completion_tokens: int
    total_cost: float
    
    def __init__(self, client: OpenAI, model: str, system_prompt: str):
        super().__init__()
        self.client = client
        self.model = model
        self.system_prompt = system_prompt
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
        self.total_cost = 0.0

    def update(self, prompt_tokens: int, completion_tokens: int):
        # 모델별 가격 설정 (2025년 4월 기준)
        pricing = {
            'gpt-3.5-turbo-0125': {'prompt': 0.0005, 'completion': 0.0015},
            'o3-mini': {'prompt': 0.0011, 'completion': 0.0044},
        }

        if self.model not in pricing:
            raise ValueError(f"모델 {self.model}에 대한 가격 정보가 없습니다.")

        prompt_cost_per_token = pricing[self.model]['prompt']
        completion_cost_per_token = pricing[self.model]['completion']

        # 누적 토큰 수 업데이트
        self.total_prompt_tokens += prompt_tokens
        self.total_completion_tokens += completion_tokens

        # 현재 호출의 비용 계산
        prompt_cost = (prompt_tokens / 1000) * prompt_cost_per_token
        completion_cost = (completion_tokens / 1000) * completion_cost_per_token
        current_cost = prompt_cost + completion_cost

        # 총 비용 업데이트
        self.total_cost += current_cost

        # 현재 호출의 비용과 누적 비용 출력
        # print(f"현재 호출 비용: ${current_cost:.6f} USD")
        # print(f"누적 비용: ${self.total_cost:.6f} USD")
        log_entry = (
            f"Prompt tokens: {prompt_tokens}, Completion tokens: {completion_tokens}, "
            f"Current Cost: ${current_cost:.6f}, Total Cost: ${self.total_cost:.6f}\n"
        )

        # 로그 파일에 기록 (append mode)
        with open("log/planning_api_usage_log.txt", "a", encoding="utf-8") as log_file:
            log_file.write(log_entry)
        
    def __call__(self, queries: List[str], **kwargs)->List[Dict[str, str]]:
        responses = []
        for query in queries:
            prompt = f"{query}"
            completion = self.client.chat.completions.create(
                model = self.model,
                messages = [
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": prompt}
                ],
                # **kwargs
            )
            # print('## hj153lee ##')            
            # print(prompt)
            usage = completion.usage            
            prompt_tokens = usage.prompt_tokens
            completion_tokens = usage.completion_tokens

            # 토큰 및 비용 업데이트
            self.update(prompt_tokens, completion_tokens)
            
            resp = {'text': completion.choices[0].message.content, 'finish_reason': completion.choices[0].finish_reason}            
            responses.append(resp)            
        
        return responses
#!pip install google-generativeai
import os
from abc import ABC, abstractmethod
from datetime import datetime

import google.generativeai as genai  # Google Generative AI SDK&#8203;:contentReference[oaicite:0]{index=0}

class GenerateResponseBase(ABC):
    """Abstract base class for generating responses from a language model."""
    @abstractmethod
    def generate_responses(self, queries):
        """Generate responses for a list of query strings."""
        pass

class GoogleGenerateResponse(GenerateResponseBase):
    """
    A generator class for Google Gemini 2.0 Flash and Flash-Lite models via the Google Generative AI SDK.
    
    This class is structured similarly to OpenAiGenerateResponse, but uses Google's Gemini 2.0 Flash and Flash-Lite models. 
    It retrieves the API key from an environment variable and uses the official `google.generativeai` SDK to generate text.
    It tracks token usage and cost per request (based on April 2025 pricing) and logs these to a file.
    """
    def __init__(self, model_name: str = "gemini-2.0-flash"):
        # Configure API key from environment (e.g., "GEMINI_API_KEY")&#8203;:contentReference[oaicite:1]{index=1}
        api_key = os.environ.get("GEMINI_API_KEY")        
        if not api_key:
            raise RuntimeError("Google Generative AI API key not found in environment variable 'GEMINI_API_KEY'.")
        genai.configure(api_key=api_key)  # Set the API key for the SDK&#8203;:contentReference[oaicite:2]{index=2}
        
        # Determine model and pricing rates (USD per token) based on latest pricing (April 2025)
        # Gemini 2.0 Flash: $0.10 per 1M input tokens, $0.40 per 1M output tokens&#8203;:contentReference[oaicite:3]{index=3}
        # Gemini 2.0 Flash-Lite: $0.075 per 1M input tokens, $0.30 per 1M output tokens&#8203;:contentReference[oaicite:4]{index=4}
        if model_name not in ("gemini-2.0-flash", "gemini-2.0-flash-lite"):
            raise ValueError("Unsupported model_name. Use 'gemini-2.0-flash' or 'gemini-2.0-flash-lite'.")
        self.model_name = model_name
        if model_name == "gemini-2.0-flash":
            self.input_rate = 0.0001 / 1000
            self.output_rate = 0.0004 / 1000
        elif model_name == "gemini-2.0-flash-lite":  # gemini-2.0-flash-lite
            self.input_rate = 0.000075 / 1000
            self.output_rate = 0.0003 / 1000

        # Initialize the model instance using the SDK
        self.model = genai.GenerativeModel(model_name)  # Create model object&#8203;:contentReference[oaicite:5]{index=5}
        
        # Initialize counters for cumulative usage
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_cost = 0.0

    def generate_responses(self, queries: list[str]) -> list[dict]:
        """
        Generate responses for each query in the list using the configured Gemini model.
        
        Returns a list of dictionaries, each containing 'text' (the model's response) and 'finish_reason'.
        Also prints and logs token usage and cost for each query.
        """
        responses = []
        for query in queries:
            # Call the model to generate content for the query&#8203;:contentReference[oaicite:6]{index=6}
            result = self.model.generate_content(query)  # perform the API call
            text = result.text  # the generated text content of the first candidate&#8203;:contentReference[oaicite:7]{index=7}
            # Obtain the finish reason if available (why generation stopped)
            finish_reason = getattr(result, "finish_reason", None)
            if finish_reason is None and hasattr(result, "candidates"):
                # If finish_reason not a direct attribute, get from first candidate if present
                try:
                    finish_reason = result.candidates[0].finish_reason
                except Exception:
                    finish_reason = None

            # Get token usage from response usage metadata&#8203;:contentReference[oaicite:8]{index=8}
            prompt_tokens = 0
            output_tokens = 0
            usage_meta = getattr(result, "usage_metadata", None)
            if usage_meta:
                # The usage_metadata provides token counts for prompt and response
                if hasattr(usage_meta, "prompt_token_count"):
                    prompt_tokens = getattr(usage_meta, "prompt_token_count")
                    output_tokens = getattr(usage_meta, "candidates_token_count", 0)
                elif hasattr(usage_meta, "promptTokenCount"):  # if for some reason camelCase
                    prompt_tokens = getattr(usage_meta, "promptTokenCount")
                    output_tokens = getattr(usage_meta, "candidatesTokenCount", 0)
            # If usage_metadata is not available or tokens are 0, use count_tokens as fallback
            if not usage_meta or (prompt_tokens == 0 and output_tokens == 0):
                try:
                    # Use the SDK's token counting for precise measure (prompt)&#8203;:contentReference[oaicite:9]{index=9}
                    count_resp = self.model.count_tokens(query)
                    if hasattr(count_resp, "token_count"):
                        prompt_tokens = count_resp.token_count
                    else:
                        prompt_tokens = getattr(count_resp, "tokenCount", len(query))  # fallback to length if needed
                except Exception:
                    prompt_tokens = len(query)  # fallback: approximate tokens by length
                try:
                    # Count output tokens similarly
                    count_resp_out = self.model.count_tokens(text)
                    if hasattr(count_resp_out, "token_count"):
                        output_tokens = count_resp_out.token_count
                    else:
                        output_tokens = getattr(count_resp_out, "tokenCount", len(text))
                except Exception:
                    output_tokens = len(text)
            
            # Calculate cost for this request using pricing rates
            cost = prompt_tokens * self.input_rate + output_tokens * self.output_rate
            # Update cumulative totals
            self.total_input_tokens += prompt_tokens
            self.total_output_tokens += output_tokens
            self.total_cost += cost

            # Print token usage and cost for this query
            print(f"Model: {self.model_name}, Prompt tokens: {prompt_tokens}, Output tokens: {output_tokens}, "
                  f"Cost: ${cost:.6f}, Cumulative cost: ${self.total_cost:.6f}")
            # Log the usage and cost to a file
            with open("log/planning_api_usage_log.txt", "a") as log_file:
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                log_file.write(
                    f"{timestamp} - Model: {self.model_name}, Prompt tokens: {prompt_tokens}, "
                    f"Output tokens: {output_tokens}, Cost: ${cost:.6f}, "
                    f"Total prompt tokens so far: {self.total_input_tokens}, "
                    f"Total output tokens so far: {self.total_output_tokens}, "
                    f"Cumulative cost so far: ${self.total_cost:.6f}\n"
                )

            # Store the response text and finish_reason in the result list
            #responses.append({"text": text, "finish_reason": finish_reason})
            responses.append({"text": text})
        return responses


In [4]:
ZERO_SHOT_INFERENCE_GENERATION_PROMPT = """
Please help identify the appropriate function for the query and assist in generating the parameters. I will provide descriptions of several tools.  
Based on these tools, you must generate a response to the given query. In other words, the response should be written in a way that calls the appropriate tool to fulfill the user's request. The requirements are as follows:

1. For each query, you must choose the appropriate tool from the given list and provide a response. This means you must specify the value of each parameter used in the selected tool.
2. If the query does not match any of the provided tools, set the tool name to "None".
3. If a parameter has `required=False`, you may omit it. However, if the query implicitly includes information corresponding to that parameter, you must include its value.
4. The generated data must follow the format of the example provided for that tool.
5. All parameter values used in the function call must be inferable from the user's request. Do not make up values that are not present or implied in the query.

Below is the information on the tools I provided:
Tools: {tools}

Now, please generate a response to the query below. The generated data must include both the query and the correct answer key which include tool and parameters inside. 
format: {re_format}
The result should be in JSON format, following the same structure as the format above.    
Do not create any parameter values that cannot be inferred from the user’s query.
{data}
"""

ZERO_SHOT_HISTORY_INFERENCE_GENERATION_PROMPT = """
Please help identify the appropriate function for the query and assist in generating the parameters. I will provide descriptions of several tools.  
Based on these tools, you must generate a response to the given query. In other words, the response should be written in a way that calls the appropriate tool to fulfill the user's request. The requirements are as follows:

1. For each query, you must choose the appropriate tool from the given list and provide a response. This means you must specify the value of each parameter used in the selected tool.
2. If the query does not match any of the provided tools, set the tool name to "None".
3. If a parameter has `required=False`, you may omit it. However, if the query implicitly includes information corresponding to that parameter, you must include its value.
4. The generated data must follow the format of the example provided for that tool.
5. All parameter values used in the function call must be inferable from the user's request. Do not make up values that are not present or implied in the query.
6. You may refer to the conversation_history to generate an appropriate response.

Below is the information on the tools I provided:
Tools: {tools}

Now, please generate a response to the query below. The generated data must include both the query and the correct answer key which include tool and parameters inside. 
format: {re_format}
The result should be in JSON format, following the same structure as the format above.    
Do not create any parameter values that cannot be inferred from the user’s query.

conversation_history: {conversation_history}
{data}
"""

FEW_SHOT_INFERENCE_GENERATION_PROMPT = """
Please help identify the appropriate function for a given query and assist in generating its parameters.  
I will provide descriptions and example data for several tools.  
Based on these tools, you must generate a response to the given query. In other words, the response should be written in a way that calls the appropriate tool to fulfill the user's request. The requirements are as follows:

1. For each query, you must choose the appropriate tool from the given list and provide a response. This means you must specify the value of each parameter used in the selected tool.
2. If the query does not match any of the provided tools, set the tool name to "None".
3. If a parameter has `required=False`, you may omit it. However, if the query explicitly includes information corresponding to that parameter, you must include its value.
4. The generated data must follow the format of the example provided for that tool.
5. All parameter values used in the function call must be inferable from the user's request. Do not make up values that are not present or implied in the query.

Below are examples of tool descriptions and query-response pairs that I’ve provided:  
Tool-example pairs: {tools-examples}

Now, please generate a response to the query below. The generated data must include both the query and the correct answer key.  
The result should be in JSON format, following the same structure as the examples above.  
Do not create any parameter values that cannot be inferred from the user’s query.
{data}
"""

In [46]:
### baseline ###
### baseline ###
### baseline ###
### baseline ###
### baseline ###

import json
import random
from collections import defaultdict

# 1. 두 jsonl 파일을 로드합니다.
#target_file = "data/step3_iter1.jsonl"
target_file = "data/step1_iter1.jsonl"
api_file = "data/api_v3.0.1.jsonl"
examples_file = 'data/step1_iter1.jsonl' # 예시는 플랜 별로 잘 된 케이스들을 별도로 관리해야 할 듯

prompt_template = ZERO_SHOT_HISTORY_INFERENCE_GENERATION_PROMPT

def extract_samples(samples, samples_per_group=5):
    groups = defaultdict(list)
    for data in samples:
        try:            
            # answer.name 값을 가져옴
            answer_name = data.get("answer", {}).get("name")
            if answer_name:
                groups[answer_name].append(data)
        except json.JSONDecodeError as e:
            print("JSON 파싱 오류로 해당 줄 건너뜀:", e)
    
    # 각 그룹별로 최대 5개씩 추출
    result = {}
    for answer_name, items in groups.items():
        result[answer_name] = items[:samples_per_group]
        
    return result
    
target_datas = []
with open(target_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            target_datas.append(json.loads(line))

extracted_samples = extract_samples(target_datas)
examples_list = []
with open(examples_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            examples_list.append(json.loads(line))

# 2. api.jsonl 파일의 각 항목을 key가 name인 딕셔너리에 저장합니다.
api_dict = {}
with open(api_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            entry = json.loads(line)
            api_dict[entry["name"]] = entry

def filter_data_list_by_name(data_list, target_name):
    """
    JSON 객체들이 담긴 리스트에서 각 객체의 answers[0].name이 target_name과 일치하는 항목을 5개 찾아서,
    query와 answer (첫 번째 answer 객체)만 포함하는 리스트를 반환합니다.
    
    :param data_list: JSON 객체들이 담긴 리스트
    :param target_name: 비교할 answer의 name 값
    :return: query와 answer 키만 포함된 dict들의 리스트
    """
    results = []
    for record in data_list:
        # answers 리스트가 존재하며 최소 1개 이상의 요소가 있는지 확인
        answer = record.get("answer", [])
        if answer and answer.get("name") == target_name:
            results.append({
                "query": record.get("query", ""),
                "answer": answer  # 첫번째 answer 전체를 포함
            })
            if len(results) == 1:
                break
    return results


prompts_info_base = []
def convert_api_compacted(api_data, zero_shot=True):    
    if zero_shot: # if zero-shot
        api_data.pop("examples", None)
    api_data.pop("returns", None)
    api_data.pop("next_turn_functions", None)

#for target_data in target_datas: # All
random.seed(42)
for key in extracted_samples: # Sampling 5 per each
    target_datas = extracted_samples[key] # Sampling 5 per each
    for target_data in target_datas:
        ground_truth = {
            "name": target_data["answer"]["name"],
            "arguments": target_data["answer"]["arguments"]
        }    
    
        # sample N-1 random functions from api_dict plus ground truth
        N = 5
        gt_func_name = ground_truth["name"]
        random_keys = random.sample([k for k in api_dict if k != gt_func_name], N-1)
        random_keys.append(gt_func_name)
        random.shuffle(random_keys)
    
        api_datas = ""
        for func_key in random_keys:
            api_data = api_dict[func_key].copy()
            convert_api_compacted(api_data, zero_shot=True)
            api_datas += json.dumps(api_data, indent=2, ensure_ascii=False) + "\n"

        #conv_history = json.dumps({"conversation_history": target_data["conversation_history"]}, indent=2, ensure_ascii=False)
        test_data = json.dumps({"query": target_data["query"]}, indent=2, ensure_ascii=False)
    
        examples = filter_data_list_by_name(examples_list, gt_func_name)
        examples = [json.dumps(ex, indent=2, ensure_ascii=False) for ex in examples]
        examples_ = "\n".join(examples)
        
        re_format = {"query": "str type query", "answer": {"tool": "str type tool", "parameters": "dict type parameters"}}
        prompt_text = prompt_template.format(            
            tools = api_datas,
            data = test_data,
            conversation_history = [],
            re_format = re_format
            #examples = examples_
        )
        target_data["candidates"] = random_keys
        prompts_info_base.append({
            "prompt": prompt_text,
            "gt": target_data
            })
                                                        


In [27]:
# path = 'evaluation/p0_baseline_n5.txt'
# with open(path, "w", encoding="utf-8") as file:
#     json.dump(prompts_info, file)

In [41]:
path = 'evaluation/p0_baseline_n5.txt'
with open(path, "r", encoding="utf-8") as file:
    prompts_info = json.load(file)

In [42]:
print(prompts_info[4]["prompt"])
#len(prompts_info)


Please help identify the appropriate function for the query and assist in generating the parameters. I will provide descriptions of several tools.  
Based on these tools, you must generate a response to the given query. In other words, the response should be written in a way that calls the appropriate tool to fulfill the user's request. The requirements are as follows:

1. For each query, you must choose the appropriate tool from the given list and provide a response. This means you must specify the value of each parameter used in the selected tool.
2. If the query does not match any of the provided tools, set the tool name to "None".
3. If a parameter has `required=False`, you may omit it. However, if the query implicitly includes information corresponding to that parameter, you must include its value.
4. The generated data must follow the format of the example provided for that tool.
5. All parameter values used in the function call must be inferable from the user's request. Do not 

In [44]:
import json
from utils import JsonExtractor, SimilarityFilter, DataFilter

model = ['o3-mini', 'gem-f', 'gem-f-l']
testtype = ['baseline', 'ours']
candidates_num = [5, 10]

model_name = model[1]

o_path = f'evaluation/result-p0-{model_name}-{testtype[0]}-{candidates_num[0]}'
print(o_path)
output_file = open(o_path, "w")
MODEL_CLASS_MAP = {
    "gpt": {
        "base_url": "https://api.openai.com/v1",
        "api_key": os.environ.get("OPENAI_API_KEY", None),
    },
    "deepseek": {
        "base_url": "https://api.deepseek.com/",
        "api_key": os.environ.get("DEEPSEEK_API_KEY", None),
    }
}

client = OpenAI(api_key=MODEL_CLASS_MAP["gpt"]["api_key"], base_url=MODEL_CLASS_MAP["gpt"]["base_url"])
generate_response = OpenAiGenerateResponse(client=client, model="o3-mini", system_prompt="")
if model_name == 'gem-f':
    gemini_generate_response = GoogleGenerateResponse(model_name="gemini-2.0-flash")
elif model_name == 'gem-f-l':
    gemini_generate_response = GoogleGenerateResponse(model_name="gemini-2.0-flash-lite")
filters = [JsonExtractor()]

response_datasets = []
for i, data in enumerate(prompts_info):        
    prompt = data["prompt"]
    gt = data["gt"]
    try:
        if model_name != 'o3-mini':
            response = gemini_generate_response.generate_responses([prompt])
        else:
            response = generate_response([prompt])        
        for filter_idx, filter in enumerate(filters):
            responses = filter.filter(response)
            
        for idx, res in enumerate(responses):            
            response_datasets.append(res)
            gt["generated"] = res
            gt["results"] = {
                "tool": gt["answer"]["name"] == res["answer"]["tool"],
                "arguments": gt["answer"]["arguments"] == res["answer"]["parameters"]
            }
            output_file.write(json.dumps(gt, ensure_ascii=False)+"\n")
            output_file.flush()    
            print(gt["results"])
    
        print(f"--- Prompt {i+1} / {len(prompts_info)}, len(datasets): {len(response_datasets)} ---")    
    except Exception as e:
        print(f'error: {e}, index: {i}')
        gt["generated"] = res
        gt["results"] = {"tool": "exception", "arguments": "exception"}
        output_file.write(json.dumps(gt, ensure_ascii=False)+"\n")
        output_file.flush()    
            


evaluation/result-p0-gem-f-baseline-5
Model: gemini-2.0-flash, Prompt tokens: 1160, Output tokens: 88, Cost: $0.000151, Cumulative cost: $0.000151
{'tool': True, 'arguments': True}
--- Prompt 1 / 205, len(datasets): 1 ---
Model: gemini-2.0-flash, Prompt tokens: 1426, Output tokens: 91, Cost: $0.000179, Cumulative cost: $0.000330
{'tool': True, 'arguments': True}
--- Prompt 2 / 205, len(datasets): 2 ---
Model: gemini-2.0-flash, Prompt tokens: 1255, Output tokens: 83, Cost: $0.000159, Cumulative cost: $0.000489
{'tool': True, 'arguments': True}
--- Prompt 3 / 205, len(datasets): 3 ---
Model: gemini-2.0-flash, Prompt tokens: 1035, Output tokens: 82, Cost: $0.000136, Cumulative cost: $0.000625
{'tool': True, 'arguments': True}
--- Prompt 4 / 205, len(datasets): 4 ---
Model: gemini-2.0-flash, Prompt tokens: 1164, Output tokens: 87, Cost: $0.000151, Cumulative cost: $0.000776
{'tool': True, 'arguments': True}
--- Prompt 5 / 205, len(datasets): 5 ---
Model: gemini-2.0-flash, Prompt tokens: 14